# Graphene Nanoribbon Density of States

This notebook builds a small bipartite honeycomb nanoribbon tight-binding model. A Gaussian polynomial energy window estimates the local density of states near selected energies.

The model is finite and deliberately small, but it captures the spectral filtering task used in graphene and nanoribbon calculations.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from qsvt.matrix_functions import design_gaussian_window_polynomial
from qsvt.polynomials import eval_polynomial
from qsvt.rescaling import rescale_hermitian_to_unit_interval
from qsvt.spectral import apply_polynomial_to_hermitian, eigh_hermitian

np.set_printoptions(precision=4, suppress=True)

In [ ]:
def graphene_ribbon_hamiltonian(nx, ny, hopping=1.0):
    def index(x, y, sublattice):
        return 2 * (y * nx + x) + sublattice

    n = 2 * nx * ny
    H = np.zeros((n, n), dtype=float)
    for x in range(nx):
        for y in range(ny):
            a = index(x, y, 0)
            neighbors = [(x, y), (x - 1, y), (x, y - 1)]
            for xb, yb in neighbors:
                if 0 <= xb < nx and 0 <= yb < ny:
                    b = index(xb, yb, 1)
                    H[a, b] = H[b, a] = -hopping
    return H


nx, ny = 6, 4
H = graphene_ribbon_hamiltonian(nx, ny)
evals, evecs = eigh_hermitian(H)
scaled = rescale_hermitian_to_unit_interval(H)
scaled_evals = np.linalg.eigvalsh(scaled.matrix)

zero_center = (0.0 - scaled.offset) / scaled.scale
window_width = 0.09
coeffs = design_gaussian_window_polynomial(zero_center, window_width, degree=34)
window_operator = apply_polynomial_to_hermitian(scaled.matrix, coeffs)
zero_window_weights = eval_polynomial(coeffs, scaled_evals)
local_dos = np.real(np.diag(window_operator))

edge_sites = []
for x in range(nx):
    for sub in [0, 1]:
        edge_sites.extend([2 * x + sub, 2 * ((ny - 1) * nx + x) + sub])
edge_sites = np.asarray(edge_sites)
edge_fraction = local_dos[edge_sites].sum() / local_dos.sum()
edge_fraction

In [ ]:
centers = np.linspace(evals.min(), evals.max(), 90)
dos = []
for center in centers:
    center_scaled = (center - scaled.offset) / scaled.scale
    c = design_gaussian_window_polynomial(center_scaled, window_width, degree=26)
    dos.append(np.sum(eval_polynomial(c, scaled_evals)))
dos = np.asarray(dos)

fig, axes = plt.subplots(1, 3, figsize=(12, 3.6), constrained_layout=True)

axes[0].scatter(np.arange(len(evals)), evals, s=18)
axes[0].axhline(0.0, color="0.4", linestyle="--")
axes[0].set_xlabel("eigenstate index")
axes[0].set_ylabel("energy")
axes[0].set_title("Nanoribbon spectrum")

axes[1].plot(centers, dos)
axes[1].axvline(0.0, color="0.4", linestyle="--")
axes[1].set_xlabel("energy")
axes[1].set_ylabel("windowed DOS")
axes[1].set_title("Gaussian spectral windows")

image = local_dos.reshape(ny, nx, 2).sum(axis=2)
im = axes[2].imshow(image, origin="lower", aspect="auto", cmap="magma")
axes[2].set_xlabel("unit cell x")
axes[2].set_ylabel("unit cell y")
axes[2].set_title("near-zero local DOS")
fig.colorbar(im, ax=axes[2], fraction=0.046)

plt.show()

In [ ]:
assert H.shape == (2 * nx * ny, 2 * nx * ny)
assert np.allclose(H, H.T)
assert np.max(np.abs(evals + evals[::-1])) < 1e-10
assert np.sum(zero_window_weights) > 1.0
assert edge_fraction > 0.35

print(f"near_zero_window_weight: {np.sum(zero_window_weights):.3f}")
print(f"edge_fraction_of_near_zero_ldos: {edge_fraction:.3f}")
print("validation: passed")